# Hydro case benchmark: SSP5-8.5 vs SSP1-1.9

This notebook computes benchmark rows for `dpRF` and `dpAGWP` from the four provided pickle files.

Definition used (as requested):

\[
\text{Year of maximum absolute gap} = \arg\max_t\left|\text{SSP5-8.5}(t) - \text{SSP1-1.9}(t)\right|
\]

\[
\text{Difference vs SSP1-1.9} = \frac{\text{SSP5-8.5} - \text{SSP1-1.9}}{\text{SSP1-1.9}}\times 100\%
\]

Note: despite the label "absolute gap", the implementation follows your formula `max_t(SSP585 - SSP119)` (not max of absolute value).



In [ ]:
import pickle
import numpy as np
import pandas as pd
BASE = '/Users/susierwu/dpLCA_main/Elec_case/dpLCI_v2026/dpLCIA_hydro/results_hydro_dpLCIA'
FILES = {
    ('dpRF', 2030): 'results_rf_2030.pkl',
    ('dpRF', 2050): 'results_rf_2050.pkl',
    ('dpAGWP', 2030): 'results_agwp_2030.pkl',
    ('dpAGWP', 2050): 'results_agwp_2050.pkl',
}
FLOW_MAP = {
    'all_ghg': 'Total GHG',
    'co2_total': 'CO2 total',
    'ch4_total': 'CH4 total',
    'n2o_total': 'N2O total',
}
class SafeUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module.startswith('dpLCIA'):
            return type(name, (), {})
        return super().find_class(module, name)
def load_pickle(path):
    with open(path, 'rb') as f:
        return SafeUnpickler(f).load()
def build_benchmark_table():
    rows = []
    for (metric, model_year), fn in FILES.items():
        obj = load_pickle(f'{BASE}/{fn}')
        s119 = obj[('SSP1-19', model_year)]['central']
        s585 = obj[('SSP5-85', model_year)]['central']
        years = s119.index.year
        for col, flow_label in FLOW_MAP.items():
            diff = s585[col] - s119[col]
            i = int(diff.abs().values.argmax())
            year = int(years[i])
            v119 = float(s119[col].iloc[i])
            v585 = float(s585[col].iloc[i])
            d = float(diff.iloc[i])
            pct = np.nan if v119 == 0 else (d / v119 * 100.0)
            rows.append({
                'Metric': metric,
                'ModelYear': model_year,
                'Flow category': flow_label,
                'Year of maximum absolute gap': year,
                'SSP1-1.9 value': v119,
                'SSP5-8.5 value': v585,
                'SSP5-8.5 - SSP1-1.9': d,
                'Difference vs. SSP1-1.9 (%)': pct,
            })
    df = pd.DataFrame(rows)
    metric_order = pd.CategoricalDtype(['dpRF', 'dpAGWP'], ordered=True)
    flow_order = pd.CategoricalDtype(['Total GHG', 'CO2 total', 'CH4 total', 'N2O total'], ordered=True)
    df['Metric'] = df['Metric'].astype(metric_order)
    df['Flow category'] = df['Flow category'].astype(flow_order)
    df = df.sort_values(['ModelYear', 'Metric', 'Flow category']).reset_index(drop=True)
    df['Metric'] = df['Metric'].astype(str)
    df['Flow category'] = df['Flow category'].astype(str)
    return df
benchmark = build_benchmark_table()
benchmark


In [ ]:
benchmark_pretty = benchmark.copy()
for c in ['SSP1-1.9 value', 'SSP5-8.5 value', 'SSP5-8.5 - SSP1-1.9']:
    benchmark_pretty[c] = benchmark_pretty[c].map(lambda x: f'{x:.3e}')
benchmark_pretty['Difference vs. SSP1-1.9 (%)'] = benchmark_pretty['Difference vs. SSP1-1.9 (%)'].map(
    lambda x: 'NA' if pd.isna(x) else f'{x:+.1f}%'
)
benchmark_pretty


In [ ]:
co2 = benchmark[benchmark['Flow category'] == 'CO2 total'].copy()
co2_biggest = co2.loc[co2['Difference vs. SSP1-1.9 (%)'].abs().idxmax()]
ch4_rf = benchmark[(benchmark['Flow category'] == 'CH4 total') & (benchmark['Metric'] == 'dpRF')].copy()
ch4_rf_biggest = ch4_rf.loc[ch4_rf['Difference vs. SSP1-1.9 (%)'].abs().idxmax()]
ch4_agwp = benchmark[(benchmark['Flow category'] == 'CH4 total') & (benchmark['Metric'] == 'dpAGWP')].copy()
print('CO2 biggest % gap (across dpRF + dpAGWP, both model years):')
print(co2_biggest.to_string())
print('
CH4 biggest % gap on dpRF:')
print(ch4_rf_biggest.to_string())
if ch4_agwp['Difference vs. SSP1-1.9 (%)'].notna().any():
    ch4_agwp_biggest = ch4_agwp.loc[ch4_agwp['Difference vs. SSP1-1.9 (%)'].abs().idxmax()]
    print('
CH4 biggest % gap on dpAGWP:')
    print(ch4_agwp_biggest.to_string())
else:
    print('
CH4 on dpAGWP: percent difference is NA for the selected max-gap year(s) because SSP1-1.9 value is 0.')


In [ ]:
out_dir = '/Users/susierwu/Documents/Codex/2026-05-08/files-mentioned-by-the-user-results'
benchmark.to_csv(f'{out_dir}/hydro_benchmark_table_raw.csv', index=False)
benchmark_pretty.to_csv(f'{out_dir}/hydro_benchmark_table_pretty.csv', index=False)
print('Saved:')
print(f'- {out_dir}/hydro_benchmark_table_raw.csv')
print(f'- {out_dir}/hydro_benchmark_table_pretty.csv')
